# Visualize Trajectory Datasets
Automatically parses randomized Minari distributions extracting specific `EpisodeData` mapping `InfRRTStar` generated routes safely integrating geometric representations globally.

In [ ]:
# PARAMETERS
DATASET_ID = "custom/mpd_largemaze_ellipsoids-v1"
NUM_SAMPLES = 5 # Episodes to visualize randomly
ENVIRONMENT_NAME = 'EnvLargeMazeRandomEllipsoids2D' 


In [ ]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import minari

sys.path.append('/home/earth/MPD/mpd-public')
sys.path.append('/home/earth/MPD/mpd-public/deps/torch_robotics')

from trajectory_generator import EnvUMazeRandomEllipsoids2D, EnvLargeMazeRandomEllipsoids2D
tensor_args = {'device': 'cpu', 'dtype': torch.float32}

# Load the mapped dataset locally:
local_data = os.path.abspath('./data')
os.environ['MINARI_DATASETS_PATH'] = local_data
dataset = minari.load_dataset(DATASET_ID)
print(f"Dataset successfully loaded possessing {dataset.total_episodes} random trajectories.")


In [ ]:
# Fetch completely randomized episodes
np.random.seed(None)
n_to_sample = min(NUM_SAMPLES, len(dataset.episode_indices))
if n_to_sample < NUM_SAMPLES:
    print(f"  Warning: only {n_to_sample} episodes available, sampling {n_to_sample} instead of {NUM_SAMPLES}")
sampled_ids = np.random.choice(dataset.episode_indices, size=n_to_sample, replace=False)

env_class = EnvLargeMazeRandomEllipsoids2D if ENVIRONMENT_NAME == 'EnvLargeMazeRandomEllipsoids2D' else EnvUMazeRandomEllipsoids2D

for ep_id in sampled_ids:
    episode = dataset[ep_id]
    
    # Environment info constants are tiled across time steps T.
    # Take the exact snapshot from the very final mapping natively bounds T=0.
    start_pos = np.array(episode.infos['start_pos'][0])
    goal_pos = np.array(episode.infos['goal_pos'][0])
    
    centers = np.array(episode.infos['ellipsoids_centers'][0])
    radii = np.array(episode.infos['ellipsoids_radii'][0])
    
    # Safely truncate the max_obstacle sequence padded zeroes out
    valid_mask = np.sum(np.abs(radii), axis=-1) > 1e-4
    centers = centers[valid_mask]
    radii = radii[valid_mask]

    # Dynamically boot an env layout container strictly for visualization matrices
    env = env_class(tensor_args=tensor_args)
    env.ellipsoids_centers = centers
    env.ellipsoids_radii = radii
    # Map the custom centers locally bounding `MultiEllipsoidField`
    env.obj_fixed_list[0].fields[-1].centers = torch.tensor(centers, **tensor_args)
    env.obj_fixed_list[0].fields[-1].radii = torch.tensor(radii, **tensor_args)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    env.render(ax)
    
    # Plot Trajectory sequence (Observations are N x 4 or N x 2)
    obs = episode.observations
    ax.plot(obs[:, 0], obs[:, 1], color='blue', linewidth=2, label='Optimized GPMP2 Trajectory')
    
    ax.scatter(start_pos[0], start_pos[1], color='green', s=100, zorder=5, label='Start')
    ax.scatter(goal_pos[0], goal_pos[1], color='red', s=100, marker='*', zorder=5, label='Goal')
    
    ax.set_title(f"Episode {ep_id}")
    ax.legend(loc='upper right')
    plt.show()
